# 🚨 Agentic AI Coding Lab — Build an Exam Rescue Agent

**Today = coding day.** Yesterday you covered the theory; today every concept becomes working code.

### Final build

A student can say:

> *"My Biology exam is in 10 days. I scored 55/100, I have 6 topics left, I can study 1 hour a day, and my confidence is 2/5. Help me."*

The finished agent can:

- calculate the student's grade;
- calculate available study time;
- decide study priority;
- search the live web for current official study resources;
- remember follow-up messages;
- stream responses;
- show tool activity;
- return structured output;
- run as a Streamlit app.

# 0. Google Colab Setup

In Colab, open the **🔑 Secrets** panel and add:

```text
OPENAI_API_KEY
```

Turn on **Notebook access**.

We will not write the key directly into code.

In [1]:
%pip install -qU langchain langchain-openai langgraph pydantic streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 54.8 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("API key loaded from Colab Secrets.")

API key loaded from Colab Secrets.


# 1. Model — `ChatOpenAI`

The model is the language/reasoning engine.

For this class we use the OpenAI-specific LangChain integration:

```python
ChatOpenAI(...)
```

We keep the first interaction simple and use only:

```python
model.invoke(...)
```

`invoke()` sends **one input** and returns **one completed model response**.

In [3]:
from langchain_openai import ChatOpenAI
import os

# Ensure the API key is stripped of any extra whitespace
openai_api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = ChatOpenAI(model="gpt-5.4-mini", openai_api_key=openai_api_key)

response = model.invoke("Give me one practical study tip for a Biology exam.")

print(response.text)

Use **active recall**: after studying a topic, close your notes and try to explain it from memory in your own words, as if teaching someone else. This works especially well for Biology because it helps you remember processes, terms, and relationships instead of just recognizing them.


## Look at what came back

The response is not just a Python string. It is an **AI message object** containing text and metadata.

In [4]:
print("TYPE:")
print(type(response))

print("\nTEXT:")
print(response.text)

print("\nUSAGE:")
print(response.usage_metadata)

TYPE:
<class 'langchain_core.messages.ai.AIMessage'>

TEXT:
Use **active recall**: after studying a topic, close your notes and try to explain it from memory in your own words, as if teaching someone else. This works especially well for Biology because it helps you remember processes, terms, and relationships instead of just recognizing them.

USAGE:
{'input_tokens': 17, 'output_tokens': 58, 'total_tokens': 75, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 🧪 Practice 1 — Build a reusable model function

Create:

```python
ask_study_coach(question)
```

### Requirements

Your function must:

1. accept a `question`;
2. call `model.invoke(question)`;
3. return only the text response;
4. work for both test questions below.

In [5]:
# PRACTICE 1

def ask_study_coach(question):
    # TODO 1: call model.invoke(...)
    response = model.invoke(question)
    # TODO 2: return only the text
    return response.text

print(ask_study_coach("What is active recall?"))

print(ask_study_coach("Why are practice questions useful before an exam?"))

**Active recall** is a learning method where you **try to retrieve information from memory without looking at your notes or the answer first**.

### Example
Instead of rereading a chapter, you ask yourself:
- “What are the main points?”
- “Can I explain this in my own words?”
- “What are the steps in this process?”

Then you check what you got right and what you missed.

### Why it works
It strengthens memory by forcing your brain to **practice remembering**, which makes the information easier to recall later.

### Common active recall techniques
- Flashcards
- Practice quizzes
- Writing everything you remember from a topic
- Teaching the material to someone else
- Closing your notes and answering questions from memory

If you want, I can also explain **active recall vs. passive review** or give you a **study routine using active recall**.
Practice questions are useful before an exam because they help you:

- **Check what you know**: You can see which topics you understand and which on

# 2. Messages

A model call can contain a **conversation**, not only a string.

Today we will use:

- `SystemMessage` → role/rules;
- `HumanMessage` → user's message;
- `AIMessage` → previous assistant response / conversation history.

When you call the model, the new response returned by the model is itself an `AIMessage`.

In [6]:
from langchain.messages import SystemMessage , HumanMessage , AIMessage

messages = [SystemMessage(
    "You are a friendly study coach for school students."
    "Give practical and concise advice."),
            HumanMessage(
                "My biology exam is in 10 days. What should i do first?")
           ]

response = model.invoke(messages)
print(type(response))
print(response.text)


<class 'langchain_core.messages.ai.AIMessage'>
Start with a **quick diagnosis** today.

### Do this first:
1. **List the chapters/topics** in your biology syllabus.
2. **Mark each topic** as:
   - Green = I know it well
   - Yellow = I know it a bit
   - Red = I don’t know it
3. **Check the exam pattern**:
   - MCQs, short answers, diagrams, long answers?
4. **Find your weakest 3 topics** and start there.

### Best first study session plan:
- **20 min:** Scan the syllabus and past papers
- **40 min:** Study the weakest topic
- **20 min:** Make short notes or flashcards
- **20 min:** Answer a few questions on it

### Important:
Don’t start by rereading everything.  
First, find what you **don’t know**, then focus there.

If you want, I can also make you a **10-day biology study plan**.


## `AIMessage` as conversation history

Here we manually provide an earlier AI response so the model can continue the conversation.

In [7]:
conversation = [
    SystemMessage(
        "you are a friendly study coach."
    ),
    HumanMessage (
        "My Biology exam is in 10 days,"
    ),
    AIMessage(
        "How many topics do you you still need to revise"
    ),
    HumanMessage(
        "What should i focus on first?"
    ),
]

## 🧪 Practice 2 — Build a multi-turn conversation

Complete the message history so the model understands all of this:

```text
Student: I scored 58% in my last Maths test.
Coach: How confident are you from 1 to 5?
Student: 2
Student: What should I focus on first?
```

### Requirements

- Use **all three** message classes: `SystemMessage`, `HumanMessage`, `AIMessage`.
- The final message must be a `HumanMessage`.
- Call `model.invoke()` with the complete list.

In [8]:
practice_messages = [
    SystemMessage("you are a practical math study coach"),
    HumanMessage("I scored 58% in my last math test."),
    AIMessage("How confide are you from 1 to 5?"),
    HumanMessage("i'd say around 2"),
    HumanMessage("What should I focus on first?"),
]


practice_response = model.invoke(practice_messages)
print(practice_response.text)

Start with the biggest score boosters:

1. **Review the test paper**
   - Find every question you missed.
   - Label each mistake:
     - **Concept gap**: didn’t know the idea
     - **Process error**: knew it but steps went wrong
     - **Careless error**: sign, arithmetic, copying

2. **Fix the most common weak topic first**
   - Pick the topic that caused the most lost marks.
   - Don’t start with the hardest chapter—start with the one that appears most and you can improve fastest.

3. **Master the basics before harder problems**
   - Make sure you can do:
     - fractions, decimals, percentages
     - rearranging formulas
     - linear equations
     - factorizing/expanding
     - graphs if they were on the test

4. **Do short daily practice**
   - 20–30 minutes each day
   - 5 easy questions + 3 medium questions on one topic
   - Check answers immediately and correct mistakes

5. **Create an error list**
   - Write down each repeated mistake.
   - Review it before every practice s

# 3. Give the App Abilities with Normal Python

Before using LangChain tools, build the logic as **ordinary Python functions**.

> A tool is still normal code underneath.

Our Exam Rescue Agent needs three deterministic abilities:

```text
1. calculate_grade()
2. calculate_study_time()
3. determine_subject_priority()
```

## Ability 1 — Calculate grade percentage

In [9]:
def calculate_grad(marks_obtained,total_marks):
  percentage = (marks_obtained/total_marks)*100
  return round(percentage,1)

print(calculate_grad(88,100))


88.0


## 🧪 Practice 3A — Calculate available study time

Build:

```python
calculate_study_time(days_until_exam, hours_per_day)
```

For 10 days × 1.5 hours/day, the answer should be 15.0.

In [10]:
# PRACTICE 3A

def calculate_study_time(days_until_exam, hours_per_day):
    # TODO: calculate total available hours
    return 0


print(calculate_study_time(10, 1.5))

# Expected: 15.0

0


## 🧪 Practice 3B — Determine study priority

Use these rules:

```text
HIGH PRIORITY:
score < 60
OR confidence <= 2
OR topics_left >= 6

MEDIUM PRIORITY:
score < 75
OR confidence == 3

LOW PRIORITY:
anything else
```

In [11]:
# PRACTICE 3B

def determine_subject_priority(score, confidence, topics_left):

    # TODO: HIGH condition

    # TODO: MEDIUM condition

    # TODO: otherwise LOW

    return "TODO"


print(determine_subject_priority(55, 2, 6))
# Expected: HIGH PRIORITY

print(determine_subject_priority(70, 4, 3))
# Expected: MEDIUM PRIORITY

print(determine_subject_priority(85, 5, 2))
# Expected: LOW PRIORITY

TODO
TODO
TODO


# 4. Turn Python Functions into LangChain Tools

A LangChain tool gives a model access to a specific Python ability.

The simplest pattern is:

```python
@tool
def function_name(...):
    """Description the model uses to understand the tool."""
```

The **name, description, and input schema** help the model decide when and how to use the tool.

In [12]:
from langchain.tools import tool

@tool
def calculate_grade(
     marks_Obtained:float,
     total_marks:float

 ) -> float :
      """Calculate a student's percentage from marks obtained and total marks."""

      percentage = (marks_Obtained/total_marks)*100
      return round(percentage,1)

In [13]:
result = calculate_grade.invoke(
    {
        "marks_Obtained":68,
        "total_marks":80

   }
)
print(result)

85.0


## 🧪 Practice 4A — Convert study time into a tool

Create a LangChain tool named:

```python
calculate_study_time
```

### Requirements

- use `@tool`;
- add type hints;
- add a useful docstring;
- return total study hours;
- test it using `.invoke()`.

In [14]:
# PRACTICE 4A

@tool

def calculate_study_time(
    days_until_exam: int,
    hours_per_day: float


) -> float:
    """ calculate the total number of study hours available before an exam."""
    time = days_until_exam*hours_per_day
    return days_until_exam*hours_per_day




# After completing the tool, test:

print(
     calculate_study_time.invoke(
         {
             "days_until_exam": 8,
             "hours_per_day": 2
         }
     )
 )

#Expected: 16

16.0


## 🧪 Practice 4B — Build the priority tool

Now turn the priority logic into a real LangChain tool.

The description must explain **what decision the tool makes**.

In [15]:
# PRACTICE 4B

# TODO: add @tool
@tool
def determine_subject_priority(
    current_score: float,
    confidence_level:int,
    unfinished_topics: int
) -> str:
    """A tool used to calculate the priority of the subject by factoring the score and other variables"""
    priority=""
    # TODO: HIGH condition
    if current_score < 60 or confidence<=2 or unfinished_topics >= 6:
        priority="HIGH PRIORITY"
    # TODO: MEDIUM condition
    elif current_score<75 or confidence==3:
        priority="MEDIUM PRIORITY"
    # TODO: otherwise LOW
    else:
        priority="LOW PRIORITY"
    return priority

## 🐞 Bug Hunt 1 — Why does the tool fail?

The tool expects:

```text
marks_obtained
total_marks
```

but the code uses:

```python
calculate_grade.invoke({
    "marks": 55,
    "total": 100
})
```

Fix only the input dictionary.

In [16]:
# BUG HUNT 1

fixed_result = calculate_grade.invoke(
    {
        "marks_Obtained": 55,
        "total_marks": 100
    }
)

print(fixed_result)

55.0


# 5. Build the First Agent

Now the model receives multiple abilities.

The agent can decide:

```text
Should I calculate a grade?
Should I calculate available time?
Should I determine priority?
Do I need more than one tool?
```

LangChain's current high-level agent API is:

```python
create_agent(...)
```

In [17]:
from langchain.agents import create_agent


SYSTEM_PROMPT = """
You are Exam Rescue, a practical study coach for school students.

Use the available tools when calculations or priority decisions are needed.

Rules:
- Do not guess calculations that a tool can perform.
- Give realistic and concise study advice.
- Do not pretend a student is guaranteed a grade.
- If useful information is missing, ask for it.
"""
study_agent = create_agent(
    model=model,
    tools=[
        calculate_grade,
        calculate_study_time,
        determine_subject_priority
    ],
    system_prompt = SYSTEM_PROMPT
)

## First agent request

The important part:

> **We do not manually call the tools.**

The agent decides.

## Inspect what the agent actually did

Every custom tool call and tool result appears in the agent's message history.

In [18]:
result = study_agent.invoke({
   "messages" : [
       {
           "role" : "user",
           "content" : (
               "My biology exam is in 7 days."
               "I scored 52%, I have 5 topics left, "
               "and I can study 2 hours per day. "
               "Give me a short rescue plan."
           )
       }
    ]
})

print(result["messages"][-1].text)

Here’s a short rescue plan for Biology:

- **Time available:** **14 hours total** before the exam.
- **Priority:** **High** — your current score is 52% and you still have 5 topics left.

### 7-day rescue plan
- **Days 1–5:** Study **1 unfinished topic per day** for about **2 hours** each.
- **Day 6:** Revise the hardest 2 topics and make a one-page summary.
- **Day 7:** Do quick recall only — diagrams, definitions, and key processes.

### How to use each 2-hour session
- **45 min:** Learn the topic
- **45 min:** Active recall / self-test
- **30 min:** Write short answers or label diagrams

### Focus on
- Definitions
- Diagrams
- Common exam questions
- Mistakes you keep repeating

If you want, I can turn this into a **day-by-day timetable** for the 7 days.


## 🧪 Practice 5 — Tool-selection lab

Run the three prepared test cases.

Your job is **not** to write new prompts.  
Inspect the returned messages and determine which tools the agent actually chose.

In [19]:
# PRACTICE 5

test_cases = [
    "I scored 42 out of 60. What percentage did I get?",

    "My exam is in 7 days and I can study 2 hours each day. "
    "How much total study time do I have?",

    "I scored 55%. I have 7 topics unfinished, my confidence is 2/5, "
    "my exam is in 8 days, and I can study 1.5 hours daily. "
    "Assess how urgent this is and how much study time I have."
]


for i, prompt in enumerate(test_cases, start=1):

    result = study_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    print(f"\nCASE {i}")

    for message in result["messages"]:

        if getattr(message, "tool_calls", None):

            for call in message.tool_calls:
                print("Tool chosen:", call["name"])
                print("Inputs:", call["args"])


CASE 1
Tool chosen: calculate_grade
Inputs: {'marks_Obtained': 42, 'total_marks': 60}

CASE 2
Tool chosen: calculate_study_time
Inputs: {'days_until_exam': 7, 'hours_per_day': 2}

CASE 3
Tool chosen: determine_subject_priority
Inputs: {'current_score': 55, 'confidence_level': 2, 'unfinished_topics': 7}
Tool chosen: calculate_study_time
Inputs: {'days_until_exam': 8, 'hours_per_day': 1.5}


## 🐞 Bug Hunt 2 — The prompt says a tool exists, but the agent cannot use it

Suppose the system prompt says:

> *"Use the study-time tool when needed."*

But the agent tool list forgets it.

Fix the list below.

In [20]:
# BUG HUNT 2

fixed_tools = [
    calculate_grade,
    # TODO: missing tool
    determine_subject_priority
]

print([tool.name for tool in fixed_tools])

['calculate_grade', 'determine_subject_priority']


# 6. Add a Built-in Tool — Web Search

Custom tools are Python functions **we wrote**.

OpenAI also provides built-in tools such as web search.

For current web information, the LangChain OpenAI integration can pass the provider tool:

```python
{"type": "web_search_preview"}
```

For this section we use the Responses API.

In [21]:
search_model = ChatOpenAI(model ="gpt-5.4-mini",use_responses_api = True)
exam_rescue_agent = create_agent(
    model = search_model,
    tools=[
        {"type": "web_search_preview"},
        calculate_grade,
        calculate_study_time,
        determine_subject_priority
    ],


    system_prompt="""
You are Exam Rescue, a practical study coach.

Use custom calculation tools for calculations and priority decisions.

Use web search only when the student needs:
- current information,
- an official syllabus,
- an official exam-board page,
- or a current external study resource.

Prefer official sources when searching.
Do not search the web for simple calculations.
"""
)


## Test the difference

### Request A
A calculation — should use a custom tool.

### Request B
Current official information — should use web search.

In [22]:
calculation_result = exam_rescue_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I got 68 out of 80. Calculate my percentage."
            }
        ]
    }
)

print("CALCULATION:")
print(calculation_result["messages"][-1].text)

CALCULATION:
Your percentage is **85%**.


In [23]:
search_result = exam_rescue_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find me a current official Cambridge International "
                    "A Level Biology syllabus or subject page."
                )
            }
        ]
    }
)

print(search_result["messages"][-1].text)

Here’s the current official Cambridge International AS & A Level Biology subject page:

**Cambridge International AS & A Level Biology (9700)** ([cambridgeinternational.org](https://www.cambridgeinternational.org/programmes-and-qualifications/cambridge-international-as-and-a-level-biology-9700/?utm_source=openai))

It lists the current syllabus overview and shows the available syllabus documents, including **2025–2027** and **2028–2030**. ([cambridgeinternational.org](https://www.cambridgeinternational.org/programmes-and-qualifications/cambridge-international-as-and-a-level-biology-9700/?utm_source=openai))

If you want, I can also pull out the key details from the latest syllabus, such as:
- paper structure,
- topic list,
- assessment objectives,
- or entry codes.


## 🧪 Practice 6 — Search vs custom-tool investigation

Run both prepared tasks.

For each result, inspect the returned messages.

Your goal is to identify:

```text
Which request needed outside/current information?
Which request needed deterministic Python?
```

In [24]:
# PRACTICE 6

practice_tasks = [
    (
        "TASK A",
        "My exam is in 12 days and I can study 1.5 hours per day. "
        "How many total hours do I have?"
    ),
    (
        "TASK B",
        "Find a current official Cambridge International page for A Level Mathematics."
    )
]


for label, task in practice_tasks:

    print("\n", label)

    result = exam_rescue_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": task}
            ]
        }
    )

    for message in result["messages"]:

        if getattr(message, "tool_calls", None):
            print("CUSTOM TOOL CALL:", message.tool_calls)

        if getattr(message, "content_blocks", None):
            for block in message.content_blocks:
                if block.get("type") == "server_tool_call":
                    print("BUILT-IN TOOL:", block.get("name"))

    print("FINAL:", result["messages"][-1].text)


 TASK A
CUSTOM TOOL CALL: [{'name': 'calculate_study_time', 'args': {'days_until_exam': 12, 'hours_per_day': 1.5}, 'id': 'call_6MSVPuhylMFfR9UWDkVPuVva', 'type': 'tool_call'}]
FINAL: You have **18 total study hours** before your exam.

Calculation: **12 days × 1.5 hours/day = 18 hours**.

 TASK B
BUILT-IN TOOL: web_search
FINAL: The current official Cambridge International page for **A Level Mathematics** is:

- **Cambridge International AS & A Level Mathematics (9709)** ([cambridgeinternational.org](https://www.cambridgeinternational.org/Images/744634-2028-2030-syllabus.pdf?utm_source=openai))

If you want, I can also open the page and pull out the latest syllabus years and download links.


# 7. Short-Term Memory

A useful study coach should support:

```text
Student:
"My Biology exam is in 10 days."

Later:
"Actually, I can study 2 hours a day."

Later:
"Update my plan."
```

The student should not need to repeat everything.

With LangChain agents, we add a **checkpointer** and reuse a **thread ID**.

In [25]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

memory_agent = create_agent(
    model = search_model,
    tools=[
        {"type": "web_search_preview"},
        calculate_grade,
        calculate_study_time,
        determine_subject_priority
    ],
    system_prompt=SYSTEM_PROMPT,
    checkpointer = memory,
)

config = {
    "configurable": {
        "thread_id": "student-1"
    }
}


In [26]:
memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "My Biology exam is in 10 days. "
                    "I have 6 topics left and my confidence is 2/5."
                )
            }
        ]
    },
    config=config
)


follow_up = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "I can study 2 hours per day. "
                    "How much study time do I have before that exam?"
                )
            }
        ]
    },
    config=config
)


print(follow_up["messages"][-1].text)

You have **20 hours** of study time before the exam.


## 🧪 Practice 7 — Prove that threads separate conversations

Create two conversations:

```text
student-A → Biology exam in 5 days
student-B → Maths exam in 14 days
```

Then ask **each thread**:

> `"How many days did I say I have?"`

The answers should remain separate.

In [27]:
# PRACTICE 7

config_a = {
    "configurable": {
        "thread_id": "student-A"
    }
}

config_b = {
    "configurable": {
        "thread_id": "student-B"
    }
}


# TODO 1:
# Invoke memory_agent with config_a and tell it:
# "My Biology exam is in 5 days."

# TODO 2:
# Invoke memory_agent with config_b and tell it:
# "My Maths exam is in 14 days."

# TODO 3:
# Ask BOTH threads:
# "How many days did I say I have?"
#
# Print both final answers.

## 🐞 Bug Hunt 3 — Why did the agent forget?

The first call used:

```python
thread_id = "student-1"
```

The follow-up accidentally used:

```python
thread_id = "student-2"
```

Fix the follow-up thread below.

In [28]:
# BUG HUNT 3

follow_up_config = {
    "configurable": {
        "thread_id": "TODO"
    }
}

print(follow_up_config)

{'configurable': {'thread_id': 'TODO'}}


# 8. Agent Streaming

Earlier, `invoke()` waited for the complete agent result.

Now we stream **agent output**.

This is where streaming becomes useful: the agent may take time to choose tools, execute them, and generate a study plan.

LangChain supports:

```python
stream_mode="messages"
```

for model-message chunks.

In [29]:
stream_input = {
    "messages": [
        {
            "role": "user",
            "content": (
                "My Biology exam is in 7 days. "
                "I scored 52%, I have 5 topics left, "
                "and I can study 2 hours per day. "
                "Give me a short rescue plan."
            )
        }
    ]
}


for chunk in study_agent.stream(
    stream_input,
    stream_mode="messages",
    version="v2"
):

    if chunk["type"] == "messages":

        token, metadata = chunk["data"]

        text = str(token.text)

        if text:
            print(text, end="", flush=True)

14.0HIGH PRIORITYYou have **14 total study hours** and Biology is a **HIGH PRIORITY** subject.

### 7-day rescue plan
- **Days 1–5:** Finish **1 topic per day**  
  - 1 hour learn/revise
  - 1 hour active recall/questions
- **Day 6:** Review all **5 topics** quickly
  - Focus on weak areas and common exam questions
- **Day 7:** Final revision only
  - Diagrams, key definitions, processes, and short notes

### Best use of your time
- Spend **most time on the 5 unfinished topics**
- After each topic, do **5–10 quick questions** or self-test
- Make a **1-page summary** of key terms, cycles, and diagrams

### In the last 2 days
- Revise **only mistakes and high-yield points**
- Avoid learning brand-new heavy content

If you want, I can turn this into a **day-by-day timetable** for the 14 hours.

## 🧪 Practice 8 — Convert `invoke()` into streaming

The code below currently waits for the full answer.

Rewrite only the **execution part** using:

```python
study_agent.stream(...)
stream_mode="messages"
version="v2"
```

Print only textual chunks.

In [30]:
# PRACTICE 8

practice_input = {
    "messages": [
        {
            "role": "user",
            "content": (
                "I have 9 days before Maths. "
                "I can study 1 hour per day. "
                "I scored 57%. Give me a short plan."
            )
        }
    ]
}


# TODO:
# Replace this normal invoke with streaming.

practice_result = study_agent.invoke(practice_input)
print(practice_result["messages"][-1].text)

You have **9 total study hours** before Maths.

### Short 9-day plan
- **Days 1–3:** Revise weak topics first
- **Days 4–6:** Do mixed practice questions
- **Day 7:** Review formulas, rules, and common mistakes
- **Day 8:** Take a timed mini-test
- **Day 9:** Light revision + rest

### Priority order
- Focus on **topics you lose marks on most**
- Spend extra time on **unfinished or confusing chapters**
- Do **practice every day**, not just reading

### Your current score
- **57%** means you’re in a medium position, so the goal is to **push accuracy up with practice**

If you want, I can turn this into a **day-by-day Maths topic plan**.


<details>
<summary>✅ Practice 8 solution</summary>

```python
for chunk in study_agent.stream(
    practice_input,
    stream_mode="messages",
    version="v2"
):

    if chunk["type"] == "messages":

        token, metadata = chunk["data"]

        text = str(token.text)

        if text:
            print(text, end="", flush=True)
```

</details>

# 9. Event Streaming — Watch the Agent Work

Normal streaming helps us see:

> **What is the agent saying?**

Event streaming can also show:

> **What is the agent doing?**

LangChain's current event-stream API uses:

```python
agent.stream_events(..., version="v3")
```

This section is slightly more advanced. Focus on reading what happens rather than memorizing every line.

In [31]:
event_input = {
    "messages": [
        {
            "role": "user",
            "content": (
                "I got 55 out of 100. "
                "My exam is in 10 days and I can study 1 hour per day. "
                "Calculate my score and available study time."
            )
        }
    ]
}


stream = study_agent.stream_events(
    event_input,
    version="v3"
)


for name, item in stream.interleave(
    "messages",
    "tool_calls"
):

    if name == "messages":

        for delta in item.text:
            print(delta, end="", flush=True)

    elif name == "tool_calls":

        print(f"\n\n🛠 TOOL: {item.tool_name}")
        print("INPUT:", item.input)

        for _ in item.output_deltas:
            pass

        print("RESULT:", item.output)

/usr/local/lib/python3.12/dist-packages/langgraph/pregel/main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
/usr/local/lib/python3.12/dist-packages/langgraph/pregel/main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)




🛠 TOOL: calculate_grade
INPUT: {'marks_Obtained': 55, 'total_marks': 100}
RESULT: content='55.0' name='calculate_grade' id='e7562aa5-4ce0-4d0f-b9b5-94ba3690f8c6' tool_call_id='call_BUfc8vQiLNFx9GuqK2kPpFZr'


🛠 TOOL: calculate_study_time
INPUT: {'days_until_exam': 10, 'hours_per_day': 1}
RESULT: content='10.0' name='calculate_study_time' tool_call_id='call_eKp5xzSNNxvXam9fgwEpXTIb'
Your score is **55%**.

Your available study time is **10 hours** total.

If you want, I can also help you make a quick 10-hour study plan.

## 🧪 Practice 9 — Build a reusable tool watcher

Complete:

```python
watch_agent_tools(agent, prompt)
```

It should:

1. start `stream_events(..., version="v3")`;
2. consume `"tool_calls"`;
3. print the tool name;
4. print the tool inputs;
5. print the final tool result.

This is real debugging practice: you are turning event streaming into a reusable inspection helper.

In [32]:
# PRACTICE 9

def watch_agent_tools(agent, prompt):

    # TODO: start stream_events

    # TODO: consume tool_calls

    # TODO: print:
    # - tool name
    # - input
    # - output

    pass


# Test after completing:
#
# watch_agent_tools(
#     study_agent,
#     "I scored 48 out of 60 and have 6 days with 2 hours per day. "
#     "Calculate my percentage and total study time."
# )

<details>
<summary>✅ Practice 9 solution</summary>

```python
def watch_agent_tools(agent, prompt):

    stream = agent.stream_events(
        {
            "messages": [
                {"role": "user", "content": prompt}
            ]
        },
        version="v3"
    )

    for call in stream.tool_calls:

        print("TOOL:", call.tool_name)
        print("INPUT:", call.input)

        for _ in call.output_deltas:
            pass

        print("OUTPUT:", call.output)
        print("------")
```

</details>

# 10. Structured Output

A chat response is great for a human.

An application often needs predictable fields.

Instead of only:

```text
"You have 10 hours available and this is high priority..."
```

we can ask the agent to return:

```text
priority
available_hours
strategy
next_action
```

LangChain agents support this through:

```python
response_format=...
```

In [33]:
from pydantic import BaseModel

class StudyPlan(BaseModel):
    priority: str
    available_hours: float
    strategy: str
    next_action: str

structured_agent = create_agent(
    model=model,
    tools=[
        calculate_grade,
        calculate_study_time,
        determine_subject_priority
    ],
    system_prompt=SYSTEM_PROMPT,
    response_format=StudyPlan
)


structured_result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "I scored 55%. My Biology exam is in 10 days. "
                    "I can study 1 hour per day. "
                    "I have 6 topics left and my confidence is 2/5. "
                    "Create my study plan."
                )
            }
        ]
    }
)

## 🧪 Practice 10 — Extend the schema

Create `StudyPlanV2` by adding:

```python
daily_goal: str
```

Then:

1. create a new agent with `response_format=StudyPlanV2`;
2. invoke it with the prepared profile;
3. print only:
   - `priority`
   - `daily_goal`
   - `next_action`

This practices the actual relationship between a schema and `response_format`.

In [34]:
# PRACTICE 10

class StudyPlanV2(BaseModel):
    priority: str
    available_hours: float
    strategy: str
    next_action: str

    # TODO: add daily_goal


# TODO:
# Create structured_agent_v2


student_profile = (
    "I scored 62%. My exam is in 6 days. "
    "I can study 2 hours daily. "
    "I have 4 topics left and confidence is 3/5."
)


# TODO:
# Invoke the new agent.
# Print:
# - priority
# - daily_goal
# - next_action

## 🐞 Bug Hunt 4 — Field mismatch

Your schema defines:

```python
available_hours: float
```

but your code tries:

```python
plan.hours
```

Fix the variable below so it contains the correct schema field name.

In [35]:
# BUG HUNT 4

field_name = "hours"

# TODO: change to the actual field name
print("Correct schema field should be:", field_name)

Correct schema field should be: hours


# 11. Coding Checkpoint

You have now coded:

| Component | What you actually did |
|---|---|
| Model | `ChatOpenAI(...)` + `invoke()` |
| Messages | Built multi-turn message history |
| Python abilities | Wrote deterministic functions |
| Tools | Used `@tool`, input schemas, `.invoke()` |
| Agent | Used `create_agent()` |
| Built-in tool | Added OpenAI web search |
| Memory | Reused a `thread_id` |
| Streaming | Streamed agent messages |
| Event streaming | Watched tool execution |
| Structured output | Used `response_format` |

Now we package the same ideas into a small interface.

# 12. Final Build — Streamlit Exam Rescue App

The final app stays intentionally simple.

```text
Subject
Score
Days until exam
Topics left
Hours/day
Confidence
Optional official-resource search
        ↓
Exam Rescue Agent
        ↓
Study plan
        ↓
Follow-up chat using memory
```

We will create three files directly from Colab:

```text
tools.py
agent.py
app.py
```

## 12.1 Create `tools.py`

In [36]:
%%writefile tools.py

from langchain.tools import tool


@tool
def calculate_grade(
    marks_obtained: float,
    total_marks: float
) -> float:
    """Calculate a student's percentage from marks obtained and total marks."""

    return round(
        (marks_obtained / total_marks) * 100,
        1
    )


@tool
def calculate_study_time(
    days_until_exam: int,
    hours_per_day: float
) -> float:
    """Calculate total study hours available before an exam."""

    return days_until_exam * hours_per_day


@tool
def determine_subject_priority(
    current_score: float,
    confidence_level: int,
    unfinished_topics: int
) -> str:
    """Determine how urgently a student should prioritize a subject."""

    if current_score < 60 or confidence_level <= 2 or unfinished_topics >= 6:
        return "HIGH PRIORITY"

    elif current_score < 75 or confidence_level == 3:
        return "MEDIUM PRIORITY"

    else:
        return "LOW PRIORITY"

Writing tools.py


## 12.2 Create `agent.py`

In [37]:
%%writefile agent.py
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from tools import (
    calculate_grade,
    calculate_study_time,
    determine_subject_priority
)


model = ChatOpenAI(
    model="gpt-5.4-mini",
    use_responses_api=True
)


memory = InMemorySaver()


agent = create_agent(
    model=model,
    tools=[
        {"type": "web_search_preview"},
        calculate_grade,
        calculate_study_time,
        determine_subject_priority
    ],
    system_prompt="""
You are Exam Rescue, a practical study coach for school students.

Use the custom tools for:
- grade calculations,
- available study time,
- study priority.

Use web search only when a student asks for a current
or official external study resource.

When web searching, prefer official exam-board or educational sources.

Give concise, practical study advice.
Do not promise grades or guaranteed outcomes.
""",
    checkpointer=memory
)


def ask_exam_rescue(message, thread_id="student-1"):

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        }
    )

    return str(result["messages"][-1].text)

Writing agent.py


## 12.3 Create `app.py`

In [38]:
%%writefile app.py

import streamlit as st

from agent import ask_exam_rescue


st.set_page_config(
    page_title="Exam Rescue",
    page_icon="🚨"
)


st.title("🚨 Exam Rescue")

st.write(
    "Tell your AI Study Coach what is happening before your exam."
)


subject = st.text_input(
    "Subject",
    value="Biology"
)

marks = st.number_input(
    "Marks obtained",
    min_value=0.0,
    value=55.0
)

total_marks = st.number_input(
    "Total marks",
    min_value=1.0,
    value=100.0
)

days = st.number_input(
    "Days until exam",
    min_value=1,
    value=10
)

topics = st.number_input(
    "Topics left",
    min_value=0,
    value=6
)

hours = st.number_input(
    "Hours you can study per day",
    min_value=0.5,
    value=1.0,
    step=0.5
)

confidence = st.slider(
    "Confidence",
    1,
    5,
    2
)

find_resource = st.checkbox(
    "Find me one current official study resource"
)


if st.button(
    "🚨 Rescue My Exam",
    type="primary"
):

    request = f"""
My subject is {subject}.
I scored {marks} out of {total_marks}.
My exam is in {days} days.
I have {topics} topics left.
I can study {hours} hours per day.
My confidence is {confidence} out of 5.

Calculate my current percentage.
Calculate my available study time.
Determine my priority.
Create a practical rescue plan.
"""

    if find_resource:
        request += """
Also search the web for one current official study resource
that could help me with this subject.
"""

    with st.spinner(
        "Building your rescue plan..."
    ):

        answer = ask_exam_rescue(
            request
        )

    st.markdown(answer)


st.divider()

st.subheader("💬 Ask a follow-up")

follow_up = st.text_input(
    "You do not need to repeat your whole profile",
    placeholder="Make tomorrow's plan lighter."
)


if st.button("Ask Coach"):

    answer = ask_exam_rescue(
        follow_up
    )

    st.markdown(answer)

Writing app.py


# 13. Run Streamlit in Colab

Colab cannot expose `localhost` directly to your browser.

We will use a temporary **Cloudflare Quick Tunnel** for the classroom demo.

> The generated URL is temporary and public. Do not enter private or sensitive information into this demo.

In [39]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("cloudflared installed")

cloudflared installed


In [40]:
!pkill -f streamlit || true
!pkill -f cloudflared || true

print("Old demo servers stopped.")

^C
^C
Old demo servers stopped.


In [41]:
import subprocess
import time
import requests


PORT = 8501


streamlit_log = open(
    "/content/streamlit.log",
    "w"
)


streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)


time.sleep(5)


health = requests.get(
    f"http://127.0.0.1:{PORT}/_stcore/health"
)


print("Status:", health.status_code)
print("Response:", health.text)

Status: 200
Response: ok


In [42]:
from pathlib import Path
import re


tunnel_log = open(
    "/content/cloudflared.log",
    "w"
)


tunnel_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)


time.sleep(8)


log_text = Path(
    "/content/cloudflared.log"
).read_text()


match = re.search(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    log_text
)


if match:

    print("🚨 Open Exam Rescue here:")
    print(match.group(0))

else:

    print(log_text)

2026-08-09T13:46:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-09T13:46:26Z INF Requesting new quick Tunnel on trycloudflare.com...



# 14. ⚡ Student Challenge — Give Exam Rescue a New Ability

## Time: 30–45 minutes

Work individually or in pairs.

You are **not rebuilding the whole app**.

Your mission:

> Add one useful new ability to the existing Exam Rescue Agent.

---

## Choose ONE ability

### Option A — Break Planner

```python
@tool
def break_planner(study_minutes: int) -> str:
```

Possible logic:

```text
<= 30 minutes   → no long break needed
31–60 minutes   → 5 minute break
61–120 minutes  → 10–15 minute break
> 120 minutes   → split into multiple sessions
```

### Option B — Daily Topic Goal

```python
@tool
def daily_topic_goal(
    topics_left: int,
    days_until_exam: int
) -> float:
```

Example:

```text
8 topics / 4 days
→ 2 topics per day
```

### Option C — Revision Method Selector

```python
@tool
def choose_revision_method(
    confidence: int,
    days_until_exam: int
) -> str:
```

Example rules:

```text
low confidence + little time
→ practice questions + active recall

high confidence
→ timed past paper

many days left
→ spaced revision
```

## Challenge requirements

Your team must complete **all six steps**:

### 1. Build it as normal Python first

Test the logic without AI.

### 2. Convert it into `@tool`

Include:

- type hints;
- a useful docstring.

### 3. Test the tool directly

Use:

```python
your_tool.invoke({...})
```

### 4. Register it with the agent

Add it to:

```python
tools=[...]
```

### 5. Test agent decision-making

Run:

- one request where the new tool **should** be used;
- one request where it **should not** be used.

Inspect agent messages or event streaming to prove what happened.

### 6. Add one related input/output to Streamlit

Example:

```text
Break Planner
→ add preferred session length

Daily Topic Goal
→ display topics/day

Revision Method
→ use confidence + days already collected
```

In [43]:

# STUDENT CHALLENGE — STEP 1
# Build your chosen ability as NORMAL PYTHON first.

def daily_topic_goal(topics_left: int,days_until_exam: int) -> float:
    return(topics_left/days_until_exam)

print(daily_topic_goal(5,10))
print(daily_topic_goal(8,2))
print(daily_topic_goal(9,3))


# TODO: test the function with at least 3 cases

0.5
4.0
3.0


In [44]:
from langchain.tools import tool

# STUDENT CHALLENGE — STEP 2
# Convert your tested Python logic into a LangChain tool.
@tool
def daily_topic_goal(topics_left: int,days_until_exam: int) -> float:
    """A tool used to calculate the amount of topics to study per day"""
    return(topics_left/days_until_exam)

print(daily_topic_goal.invoke({"topics_left": 5, "days_until_exam": 10}))
print(daily_topic_goal.invoke({"topics_left": 8, "days_until_exam": 2}))
print(daily_topic_goal.invoke({"topics_left": 9, "days_until_exam": 3}))

0.5
4.0
3.0


In [45]:
# STUDENT CHALLENGE — STEP 3
# Test your tool DIRECTLY before giving it to the agent.

result = daily_topic_goal.invoke({"topics_left":5,"days_until_exam": 10})
print(result)

0.5


In [46]:
# STUDENT CHALLENGE — STEP 4
# Add the new tool to an agent.

challenge_agent = create_agent(
     model=model,
     tools=[
         calculate_grade,
         calculate_study_time,
         determine_subject_priority,
         daily_topic_goal
     ],
     system_prompt=SYSTEM_PROMPT
 )

In [47]:
# STUDENT CHALLENGE — STEP 5
# Prove agent decision-making.

# Test A:
# A request where your tool SHOULD run.

# Test B:
# A request where your tool SHOULD NOT run.

# Inspect tool calls or use watch_agent_tools(...)

## 🐞 Final Boss Bug Hunt

Your teammate created:

```python
new_agent = create_agent(
    model=model,
    tools=[
        calculate_grade,
        calculate_study_time
    ],
    system_prompt="""
    You can calculate grades, study time,
    priority, and break schedules.
    """
)
```

They complain:

> *"The AI knows about the priority and break tools because I wrote them in the prompt, but it never calls them!"*

### Your team must:

1. explain the bug;
2. fix the agent;
3. demonstrate that the missing tool is actually called.

### Key question

> **Who decides when a tool runs? And what must happen before the agent can make that decision?**

# 15. Exit Check — Can You Explain Your Code?

Before leaving, be able to answer:

1. What does `ChatOpenAI` create?
2. What does `model.invoke()` return?
3. Why would you include an `AIMessage` in an input message list?
4. What changes when a normal Python function gets `@tool`?
5. Why is the tool's docstring important?
6. What does `create_agent()` add around the model?
7. What is the difference between a custom Python tool and OpenAI web search?
8. Why does changing the `thread_id` change what the agent remembers?
9. What does agent streaming show?
10. What does event streaming help us inspect?
11. Why would an application use structured output?

## Final mental model

```text
MODEL
= language + reasoning

MESSAGES
= conversation context

TOOL
= a specific ability

AGENT
= model + harness + tool-selection loop

MEMORY
= thread-level conversation state

STREAMING
= live output

EVENT STREAMING
= live agent/tool activity

STRUCTURED OUTPUT
= predictable fields for applications
```

# Official LangChain References

This notebook follows the current LangChain v1 APIs:

- Agents: https://docs.langchain.com/oss/python/langchain/agents
- Tools: https://docs.langchain.com/oss/python/langchain/tools
- Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory
- Streaming: https://docs.langchain.com/oss/python/langchain/streaming
- Event streaming: https://docs.langchain.com/oss/python/langchain/event-streaming
- OpenAI integration: https://docs.langchain.com/oss/python/integrations/chat/openai

Key implementation choices:

- `ChatOpenAI`
- `create_agent`
- `@tool`
- `InMemorySaver`
- OpenAI provider web search
- `response_format`
- no legacy `initialize_agent` / `AgentExecutor`